# Session 1a: Tokenizer Playground

**Course:** Language Models: ML Basics to Modern AI (BTU Cottbus, M.Sc. AI seminar)
**Session:** 1 of 4, notebook 1a of 3
**Lecture reference:** Lecture on tokenisation, subword units, and Byte-Pair Encoding.

## Learning objectives

By the end of this notebook you should be able to:

- Explain why character-level and word-level tokenisation both fail at scale, and how BPE sits between them.
- Predict, qualitatively, how a piece of text will be split by GPT-2 and BERT and account for the differences.
- Implement the inner loop of BPE training (count adjacent pairs, merge the most frequent) and apply it to a small corpus.
- Apply trained merge rules to encode unseen text, and compare the result to a production tokenizer (GPT-2).

The notebook accompanies the lecture on tokenisation. It does not re-derive the theory. It makes the theory concrete by building it.


## §1 Primer: Byte-Pair Encoding

A language model consumes a sequence of integer IDs, not raw text. The tokenizer turns a string into IDs. The choice fixes the model's vocabulary, the average length of a sequence in tokens, and what the model can represent without falling back to an unknown token. Three families dominate.

**Character-level tokenisation.** The vocabulary is the alphabet plus a few control symbols, on the order of 100 entries. Every string is representable; there is no out-of-vocabulary problem. The cost is sequence length. A 200-word paragraph occupies roughly 1000 characters, so the model has to attend across 1000 positions to read it. Since self-attention is quadratic in sequence length, the compute and memory bill rises fast. Character models also force the network to relearn that *c*, *a*, *t* spell *cat*; the unit of meaning is below the token boundary.

**Word-level tokenisation.** The vocabulary is the set of word types in the training data, typically in the hundreds of thousands. Sequences are short (one ID per word). The problem is the open-vocabulary nature of language. New words appear constantly: proper nouns, neologisms, technical terms, typos, morphological variants the training set never saw. A word-level model meets these with a single `<unk>` token, which destroys information. It also cannot share statistical strength across related forms: *play*, *plays*, *playing*, *played* sit at distinct entries and the model only relates them through co-occurrence.

**Subword tokenisation** picks a vocabulary of common whole words, common word fragments, and individual characters as a fallback. Frequent words (*the*, *and*, *is*) get one token each. Rare or unseen words get split into pieces the model has seen before. The vocabulary-versus-sequence-length tradeoff is tunable: more merges means more whole-word tokens, shorter sequences, and a larger embedding table.

**Byte-Pair Encoding (BPE).** Originally a 1994 compression algorithm. The training procedure is iterative and greedy. Start by splitting every word in the training corpus into characters (plus an end-of-word marker so *est* at the end of a word stays distinct from *est* in the middle). Count every adjacent symbol pair across the corpus, weighted by word frequency. Find the most frequent pair, replace every occurrence with a new merged symbol, and append the merge rule to an ordered list. Repeat for a target number of merges, often in the tens of thousands. The vocabulary at the end is the original character set plus every merged symbol; the merge list is the program that encodes new text.

Concretely, given a corpus where *low* appears 5 times, *lower* 2 times, *newest* 6 times, and *widest* 3 times, the first iteration finds `e s` with frequency 9 (6 from *newest*, 3 from *widest*) and replaces every `e s` with `es`. The next iteration sees `es t` with frequency 9 and merges it into `est`. Within a handful of merges the algorithm has discovered the *-est* superlative suffix without being told that suffixes exist.

Encoding an unseen word at inference replays the merge rules in training order. Each rule contracts adjacent matching symbols; rules that do not match leave the segmentation alone. The result is a deterministic segmentation that prefers the longest learned fragments.

**WordPiece** (used by BERT) is a close cousin. The merge criterion changes: instead of picking the most frequent pair, WordPiece picks the pair that maximises the likelihood of the training data under a unigram language model. The output looks similar to BPE, but middle-of-word pieces get the `##` prefix to mark continuation, which makes detokenisation unambiguous.

**SentencePiece** (used by T5, LLaMA, most multilingual models) drops the word-boundary assumption. The tokenizer treats whitespace as just another character and runs on raw text without language-specific pre-tokenisation. SentencePiece can use either BPE or a unigram language model trained by EM. The unigram option produces a distribution over segmentations and can sample alternative tokenisations during training (subword regularisation), which helps in multilingual settings.

The rest of this notebook trains BPE end-to-end on a small corpus and compares the result to GPT-2's tokenizer. From Session 1b onward the seminar calls `AutoTokenizer.from_pretrained(...)` and stops re-implementing this layer. You will know what is inside it.


In [ ]:
"""§2 Setup: imports, random seed, tokenizer downloads."""

import random
import re
import warnings
from collections import Counter

import numpy as np
from transformers import AutoTokenizer

warnings.filterwarnings("ignore")

SEED = 0
random.seed(SEED)
np.random.seed(SEED)

# GPT-2 (BPE) and BERT (WordPiece) are the two reference tokenizers used in §3 and §4.
# The first call downloads roughly 1 MB of vocab and merge files.
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)

print(f"GPT-2 vocab size: {gpt2_tok.vocab_size:,}")
print(f"BERT  vocab size: {bert_tok.vocab_size:,}")


## §3 Guided exploration: the tokenizer playground

The playground below has three parts. Part A takes arbitrary text and shows the parallel GPT-2 and BERT tokenisations side by side, with each token coloured for readability. Part B compares token counts across nine languages for the same meaning; the asymmetry is a direct consequence of which corpora the tokenizers were trained on. Part C runs the BPE merge loop on a tiny example corpus and lets you step through the merges one at a time.

The widgets are motivation. The implementation in §5 is where you reproduce the inner loop yourself.


In [ ]:
"""§3 Tokenizer playground widget.

Three parts in one cell: side-by-side tokenisation, multilingual comparison, and a
step-through view of BPE on a tiny corpus. The original notebook also showed a GPT-4
(tiktoken) lane; that has been dropped to avoid an extra dependency.
"""

import html as html_module

import ipywidgets as widgets
from IPython.display import HTML, display

# Colour palette for token rendering.
COLORS = [
    "#f87171", "#fbbf24", "#4ade80", "#22d3ee",
    "#818cf8", "#f472b6", "#fb923c", "#34d399",
    "#60a5fa", "#c084fc", "#facc15", "#86efac",
]


def render_tokens(tokens, label):
    spans = []
    for i, tok in enumerate(tokens):
        col = COLORS[i % len(COLORS)]
        text = html_module.escape(tok).replace(" ", "·").replace("\n", "↵") or "·"
        spans.append(
            f'<span style="background:{col};color:#1e293b;padding:2px 6px;border-radius:4px;'
            f'margin:1px 2px;font-family:monospace;font-size:13px;'
            f'display:inline-block;line-height:22px">{text}</span>'
        )
    count_badge = (
        f'<span style="background:#1e293b;color:white;padding:1px 7px;'
        f'border-radius:10px;font-size:12px;margin-left:6px">{len(tokens)} tokens</span>'
    )
    header = f'<div style="font-weight:600;margin:10px 0 4px">{label}{count_badge}</div>'
    body = '<div style="line-height:30px">' + " ".join(spans) + "</div>"
    return header + body


def tokenize_gpt2(text):
    # convert_ids_to_tokens preserves the BPE byte-level form including the Ġ (space) marker.
    return gpt2_tok.convert_ids_to_tokens(gpt2_tok.encode(text, add_special_tokens=False))


def tokenize_bert(text):
    return bert_tok.tokenize(text)


# --- Part A: side-by-side GPT-2 vs BERT ---------------------------------------------
text_box = widgets.Textarea(
    value="The quick brown fox jumps over the lazy dog.",
    placeholder="Type something here.",
    layout=widgets.Layout(width="98%", height="70px"),
)
_handle_a = None


def _update_a(change=None):
    if _handle_a is None:
        return
    text = text_box.value.strip()
    if not text:
        return
    parts = [
        render_tokens(tokenize_gpt2(text), "GPT-2 (BPE)"),
        render_tokens(tokenize_bert(text), "BERT (WordPiece, ## marks continuations)"),
    ]
    _handle_a.update(HTML('<div style="font-size:15px">' + "".join(parts) + "</div>"))


text_box.observe(_update_a, names="value")


# --- Part B: multilingual comparison -----------------------------------------------
PRESETS = {
    '"happiness is contagious"': {
        "English": "happiness is contagious",
        "German": "Glueck ist ansteckend",
        "Finnish": "onnellisuus on tarttuvaa",
        "Turkish": "mutluluk bulasicidir",
        "Arabic": "السعادة معدية",
        "Japanese": "幸せは伝染する",
        "Chinese": "幸福是会传染的",
        "Spanish": "la felicidad es contagiosa",
        "Polish": "szczescie jest zarazliwe",
    },
    '"the patient reported feeling anxious"': {
        "English": "the patient reported feeling anxious",
        "German": "der Patient berichtete sich aengstlich zu fuehlen",
        "Finnish": "potilas kertoi tuntevansa ahdistusta",
        "Turkish": "hasta kaygili hissettigini bildirdi",
        "Arabic": "أبلغ المريض عن شعوره بالقلق",
        "Japanese": "患者は不安を感じていると報告した",
        "Chinese": "患者报告感到焦虑",
        "Spanish": "el paciente informo sentirse ansioso",
        "Polish": "pacjent zglosil uczucie niepokoju",
    },
}

preset_dropdown = widgets.Dropdown(
    options=list(PRESETS.keys()),
    value='"happiness is contagious"',
    description="Preset:",
    layout=widgets.Layout(width="98%"),
    style={"description_width": "60px"},
)
_handle_b = None


def _update_b(change=None):
    if _handle_b is None:
        return
    phrases = PRESETS[preset_dropdown.value]
    en_count = len(tokenize_gpt2(phrases["English"]))
    rows = []
    for lang, text in phrases.items():
        toks = tokenize_gpt2(text)
        n = len(toks)
        ratio = n / en_count if en_count > 0 else 0.0
        bar = "█" * n
        rows.append(
            f'<tr style="border-bottom:1px solid #334155">'
            f'<td style="padding:6px 12px;font-weight:600">{lang}</td>'
            f'<td style="padding:6px 12px;font-family:monospace">{html_module.escape(text)}</td>'
            f'<td style="padding:6px 12px;text-align:center;font-weight:700">{n}</td>'
            f'<td style="padding:6px 12px;text-align:center">{ratio:.1f}x</td>'
            f'<td style="padding:6px 12px;font-family:monospace;font-size:11px">{bar}</td>'
            f'</tr>'
        )
    table_html = (
        '<table style="border-collapse:collapse;width:100%;font-size:14px;margin-top:8px">'
        '<thead><tr style="background:#1e293b;color:#cbd5e1">'
        '<th style="padding:8px 12px;text-align:left">Language</th>'
        '<th style="padding:8px 12px;text-align:left">Phrase</th>'
        '<th style="padding:8px 12px">Tokens</th>'
        '<th style="padding:8px 12px">vs English</th>'
        '<th style="padding:8px 12px;text-align:left">Visual</th>'
        '</tr></thead><tbody>' + "".join(rows) + "</tbody></table>"
    )
    _handle_b.update(HTML(
        '<p style="font-size:13px;color:#94a3b8;margin-bottom:4px">Tokenizer: GPT-2 (BPE)</p>'
        + table_html
    ))


preset_dropdown.observe(_update_b, names="value")


# --- Part C: BPE step-through on the classic example ------------------------------
def _get_stats(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        syms = word.split()
        for i in range(len(syms) - 1):
            pairs[(syms[i], syms[i + 1])] += freq
    return pairs


def _merge_pair(pair, vocab):
    bigram = " ".join(pair)
    merged = "".join(pair)
    return {w.replace(bigram, merged): f for w, f in vocab.items()}


def run_bpe(corpus, num_merges=25):
    word_freqs = Counter(corpus.split())
    vocab = {" ".join(list(w) + ["</w>"]): f for w, f in word_freqs.items()}
    steps = [{"vocab": dict(vocab), "merge": None, "count": None}]
    for _ in range(num_merges):
        pairs = _get_stats(vocab)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        vocab = _merge_pair(best, vocab)
        steps.append({"vocab": dict(vocab), "merge": best, "count": pairs[best]})
    return steps


DEMO_CORPUS = "low low low lower lower newest newest newest newest newest newest widest widest widest"
_steps = run_bpe(DEMO_CORPUS)

slider = widgets.IntSlider(
    value=0, min=0, max=len(_steps) - 1, description="Step:",
    layout=widgets.Layout(width="80%"),
    style={"description_width": "40px"},
    continuous_update=True,
)
_handle_c = None


def _update_c(change=None):
    if _handle_c is None:
        return
    idx = min(slider.value, len(_steps) - 1)
    step = _steps[idx]
    vocab = step["vocab"]
    merge = step["merge"]
    count = step["count"]
    if idx == 0:
        action = '<div style="padding:8px 12px;border:1px solid #34d399;border-radius:6px;margin-bottom:12px">Step 0: each word split into characters plus end-of-word marker.</div>'
    else:
        a, b = merge
        action = (
            f'<div style="padding:8px 12px;border:1px solid #60a5fa;border-radius:6px;margin-bottom:12px">'
            f'Step {idx}: merge <code>{a}</code> + <code>{b}</code> into <code>{a + b}</code> (frequency {count})</div>'
        )
    items = []
    for word_seq, freq in sorted(vocab.items(), key=lambda x: -x[1]):
        tokens = word_seq.split()
        spans = " ".join(
            f'<span style="background:{COLORS[i % len(COLORS)]};color:#1e293b;padding:1px 5px;border-radius:3px;font-family:monospace;font-size:12px">{t}</span>'
            for i, t in enumerate(tokens)
        )
        items.append(
            f'<tr><td style="padding:4px 10px">x{freq}</td>'
            f'<td style="padding:4px 10px">{spans}</td>'
            f'<td style="padding:4px 10px;font-size:11px;color:#64748b">{len(tokens)} token(s)</td></tr>'
        )
    n_unique = len({t for ws in vocab for t in ws.split()})
    table = (
        f'<div style="font-size:13px;color:#94a3b8;margin-bottom:8px">Vocabulary size: <strong>{n_unique}</strong> unique tokens</div>'
        '<table style="font-size:14px;border-collapse:collapse">'
        '<thead><tr style="background:#1e293b;color:#cbd5e1">'
        '<th style="padding:6px 10px;text-align:left">Freq</th>'
        '<th style="padding:6px 10px;text-align:left">Word (as tokens)</th>'
        '<th style="padding:6px 10px;text-align:left"></th>'
        '</tr></thead><tbody>' + "".join(items) + "</tbody></table>"
    )
    _handle_c.update(HTML(action + table))


slider.observe(_update_c, names="value")


# --- Render all three parts -------------------------------------------------------
# Each part renders its controls, then anchors its display at a stable display_id.
# Subsequent observer callbacks update the display in place via DisplayHandle.update.

def _initial_a():
    text = text_box.value.strip()
    parts = [
        render_tokens(tokenize_gpt2(text), "GPT-2 (BPE)"),
        render_tokens(tokenize_bert(text), "BERT (WordPiece, ## marks continuations)"),
    ]
    return HTML('<div style="font-size:15px">' + "".join(parts) + "</div>")


def _initial_b():
    phrases = PRESETS[preset_dropdown.value]
    en_count = len(tokenize_gpt2(phrases["English"]))
    rows = []
    for lang, text in phrases.items():
        toks = tokenize_gpt2(text)
        n = len(toks)
        ratio = n / en_count if en_count > 0 else 0.0
        bar = "\u2588" * n
        rows.append(
            f'<tr style="border-bottom:1px solid #334155">'
            f'<td style="padding:6px 12px;font-weight:600">{lang}</td>'
            f'<td style="padding:6px 12px;font-family:monospace">{html_module.escape(text)}</td>'
            f'<td style="padding:6px 12px;text-align:center;font-weight:700">{n}</td>'
            f'<td style="padding:6px 12px;text-align:center">{ratio:.1f}x</td>'
            f'<td style="padding:6px 12px;font-family:monospace;font-size:11px">{bar}</td>'
            f'</tr>'
        )
    table_html = (
        '<table style="border-collapse:collapse;width:100%;font-size:14px;margin-top:8px">'
        '<thead><tr style="background:#1e293b;color:#cbd5e1">'
        '<th style="padding:8px 12px;text-align:left">Language</th>'
        '<th style="padding:8px 12px;text-align:left">Phrase</th>'
        '<th style="padding:8px 12px">Tokens</th>'
        '<th style="padding:8px 12px">vs English</th>'
        '<th style="padding:8px 12px;text-align:left">Visual</th>'
        '</tr></thead><tbody>' + "".join(rows) + "</tbody></table>"
    )
    return HTML(
        '<p style="font-size:13px;color:#94a3b8;margin-bottom:4px">Tokenizer: GPT-2 (BPE)</p>'
        + table_html
    )


def _initial_c():
    idx = min(slider.value, len(_steps) - 1)
    step = _steps[idx]
    vocab = step["vocab"]
    if idx == 0:
        action = '<div style="padding:8px 12px;border:1px solid #34d399;border-radius:6px;margin-bottom:12px">Step 0: each word split into characters plus end-of-word marker.</div>'
    else:
        a, b = step["merge"]
        action = (
            f'<div style="padding:8px 12px;border:1px solid #60a5fa;border-radius:6px;margin-bottom:12px">'
            f'Step {idx}: merge <code>{a}</code> + <code>{b}</code> into <code>{a + b}</code> (frequency {step["count"]})</div>'
        )
    items = []
    for word_seq, freq in sorted(vocab.items(), key=lambda x: -x[1]):
        tokens = word_seq.split()
        spans = " ".join(
            f'<span style="background:{COLORS[i % len(COLORS)]};color:#1e293b;padding:1px 5px;border-radius:3px;font-family:monospace;font-size:12px">{t}</span>'
            for i, t in enumerate(tokens)
        )
        items.append(
            f'<tr><td style="padding:4px 10px">x{freq}</td>'
            f'<td style="padding:4px 10px">{spans}</td>'
            f'<td style="padding:4px 10px;font-size:11px;color:#64748b">{len(tokens)} token(s)</td></tr>'
        )
    n_unique = len({t for ws in vocab for t in ws.split()})
    table = (
        f'<div style="font-size:13px;color:#94a3b8;margin-bottom:8px">Vocabulary size: <strong>{n_unique}</strong> unique tokens</div>'
        '<table style="font-size:14px;border-collapse:collapse">'
        '<thead><tr style="background:#1e293b;color:#cbd5e1">'
        '<th style="padding:6px 10px;text-align:left">Freq</th>'
        '<th style="padding:6px 10px;text-align:left">Word (as tokens)</th>'
        '<th style="padding:6px 10px;text-align:left"></th>'
        '</tr></thead><tbody>' + "".join(items) + "</tbody></table>"
    )
    return HTML(action + table)


display(widgets.HTML("<h4 style=\"margin-top:4px\">Part A: GPT-2 versus BERT on user-pasted text</h4>"))
display(widgets.VBox([text_box]))
_handle_a = display(_initial_a(), display_id="tokenizer-part-a")

display(widgets.HTML("<h4>Part B: token count across languages, same meaning</h4>"))
display(widgets.VBox([preset_dropdown]))
_handle_b = display(_initial_b(), display_id="tokenizer-part-b")

display(widgets.HTML("<h4>Part C: BPE step-through on <code>low / lower / newest / widest</code></h4>"))
display(widgets.VBox([slider]))
_handle_c = display(_initial_c(), display_id="tokenizer-part-c")


## §4 Warm-ups

Two short exercises before the deep build. The first reads from a production tokenizer to confirm you understand the basic API. The second implements the single most important primitive in BPE training.


In [ ]:
"""§4 Warm-up 1 (exercise): count unique tokens.

Implement count_unique_tokens to return the number of distinct token IDs that the
tokenizer emits for the given text. Use tokenizer.encode and Python's set/len.
"""

from collections import Counter

from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")


def count_unique_tokens(text: str, tokenizer) -> int:
    """Return the number of distinct token IDs the tokenizer emits for `text`."""
    # TODO: implement using tokenizer.encode and Python's set/len.
    raise NotImplementedError


sample = "The quick brown fox jumps over the lazy dog. The quick brown fox is quick."
try:
    n_unique = count_unique_tokens(sample, tok)
except NotImplementedError:
    print("count_unique_tokens not implemented yet.")
else:
    print(f"Unique tokens: {n_unique}")
    print(f"Total tokens:  {len(tok.encode(sample))}")


In [ ]:
"""§4 Warm-up 2 (exercise): the inner step of BPE training.

Implement most_frequent_pair so it returns the adjacent symbol pair with the highest
total frequency across all words in word_freq. Iterate every word's adjacent pairs
and aggregate counts weighted by the word's frequency. Use collections.Counter.
"""

from collections import Counter


def most_frequent_pair(word_freq: dict) -> tuple | None:
    """Find the adjacent symbol pair with the highest total frequency across all words.

    `word_freq` maps a tuple of symbols (the current segmentation of a word) to its
    corpus frequency. Returns None when no adjacent pairs exist.
    """
    # TODO: implement
    raise NotImplementedError


example = {
    ("l", "o", "w", "</w>"): 5,
    ("l", "o", "w", "e", "r", "</w>"): 2,
    ("n", "e", "w", "e", "s", "t", "</w>"): 6,
    ("w", "i", "d", "e", "s", "t", "</w>"): 3,
}
try:
    result = most_frequent_pair(example)
except NotImplementedError:
    print("most_frequent_pair not implemented yet.")
else:
    print(result)  # expected: ("e", "s")


## §5 Deep build: BPE end-to-end on a real corpus

The widget in §3 ran BPE on a fifteen-word toy. The warm-ups gave you the most-frequent-pair primitive. The next five cells train a BPE tokenizer on a real corpus, encode an unseen string with the learned merge rules, and compare the result to GPT-2 on the same string.

The corpus is the Tiny Shakespeare text, roughly 1 MB of 17th-century English. You will see two effects: BPE quickly merges its way out of single characters into common bigrams and trigrams, and 100 merges is nowhere near enough to compete with GPT-2's roughly 50,000 merges on a corpus three orders of magnitude larger. The point of the comparison in subtask 5 is to make the scale difference concrete.

A note on engineering choices. The implementation below uses tuples of strings to represent the current segmentation of each word and a `Counter` keyed on those tuples to track frequencies. Production tokenizers (`tokenizers` library, `sentencepiece`) use linked lists or specialised arrays and run several orders of magnitude faster. The algorithm is the same.


In [ ]:
"""§5 Subtask 1: load a real corpus.

Primary path: the HuggingFace `datasets` loader for the script-free Parquet
mirror `Trelis/tiny-shakespeare` (about 1.2 MB of Shakespeare text). The
original `tiny_shakespeare` dataset on the Hub is script-based and recent
versions of `datasets` no longer execute dataset scripts by default.

Fallback: an inline excerpt of the same text. Used when no network is
available (some sandboxed environments). The fallback is much smaller so
the resulting BPE merges are less interesting, but the notebook still runs.
"""

TINY_SHAKESPEARE_FALLBACK = """First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them. Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.

Second Citizen:
Would you proceed especially against Caius Marcius?

All:
Against him first: he's a very dog to the commonalty.

Second Citizen:
Consider you what services he has done for his country?

First Citizen:
Very well; and could be content to give him good
report fort, but that he pays himself with being proud.

Second Citizen:
Nay, but speak not maliciously.

First Citizen:
I say unto you, what he hath done famously, he did
it to that end: though soft-conscienced men can be
content to say it was for his country he did it to
please his mother and to be partly proud; which he
is, even till the altitude of his virtue.

Second Citizen:
What he cannot help in his nature, you account a
vice in him. You must in no way say he is covetous.

First Citizen:
If I must not, I need not be barren of accusations;
he hath faults, with surplus, to tire in repetition.
What shouts are these? The other side o' the city
is risen: why stay we prating here? to the Capitol!

All:
Come, come.

First Citizen:
Soft! who comes here?

Second Citizen:
Worthy Menenius Agrippa; one that hath always loved the people.

First Citizen:
He's one honest enough: would all the rest were so!

MENENIUS:
What work's, my countrymen, in hand? where go you
With bats and clubs? The matter? speak, I pray you.

First Citizen:
Our business is not unknown to the senate; they have
had inkling this fortnight what we intend to do,
which now we'll show 'em in deeds. They say poor
suitors have strong breaths: they shall know we have
strong arms too.

MENENIUS:
Why, masters, my good friends, mine honest neighbours,
Will you undo yourselves?

First Citizen:
We cannot, sir, we are undone already.

MENENIUS:
I tell you, friends, most charitable care
Have the patricians of you. For your wants,
Your suffering in this dearth, you may as well
Strike at the heaven with your staves as lift them
Against the Roman state, whose course will on
The way it takes, cracking ten thousand curbs
Of more strong link asunder than can ever
Appear in your impediment. For the dearth,
The gods, not the patricians, make it, and
Your knees to them, not arms, must help.
"""


def _load_corpus():
    try:
        from datasets import load_dataset

        ds = load_dataset("Trelis/tiny-shakespeare", split="train")
        text = "\n".join(row["Text"] for row in ds)
        return text[: 1_000_000], "datasets.load_dataset(Trelis/tiny-shakespeare)"
    except Exception as err:
        print(f"datasets loader unavailable ({type(err).__name__}); using inline fallback.")
        return TINY_SHAKESPEARE_FALLBACK, "inline fallback (Coriolanus opening, public domain)"


corpus, corpus_source = _load_corpus()
print(f"Source: {corpus_source}")
print(f"Corpus size: {len(corpus):,} characters, {len(corpus.split()):,} whitespace tokens")
print()
print(corpus[:300])


In [ ]:
"""§5 Subtask 2 (exercise): initial segmentation.

Each whitespace-separated word in the input text should become a tuple of
characters followed by the end-of-word marker "</w>". Return a Counter that
maps each such tuple to its corpus frequency.
"""


def initial_segmentation(text: str) -> Counter:
    """Each whitespace-separated word becomes a tuple of characters + end-of-word marker."""
    # TODO: implement. Hint: iterate text.split(), build tuple(word) + ("</w>",), accumulate counts.
    raise NotImplementedError


try:
    word_freq = initial_segmentation(corpus)
except NotImplementedError:
    word_freq = None
    print("initial_segmentation not implemented yet.")
else:
    print(f"Unique words: {len(word_freq):,}")
    print("First five entries:")
    for symbols, freq in list(word_freq.items())[:5]:
        print(f"  {freq:>5}  {symbols}")


In [ ]:
"""§5 Subtask 3 (exercise): the merge loop.

Implement apply_merge to rebuild the segmentation after one merge rule (a, b) is
applied: scan each tuple, replace adjacent (a, b) pairs with the merged symbol
a + b, and keep frequencies unchanged.

Then run the 100-step loop: at each step call most_frequent_pair (from warm-up 2),
apply the merge, and record it. Log progress every ten iterations.
"""


def apply_merge(word_freq: Counter, pair: tuple) -> Counter:
    """Return a new word_freq with every occurrence of `pair` merged into one symbol."""
    # TODO: implement.
    raise NotImplementedError


merges: list = []

try:
    if word_freq is None:
        raise RuntimeError("Run subtask 2 first to produce `word_freq`.")
    current = word_freq
    for step in range(100):
        pair = most_frequent_pair(current)
        if pair is None:
            break
        current = apply_merge(current, pair)
        merges.append(pair)
        if step % 10 == 0:
            print(f"step {step:3d}: merged {pair}")
    print(f"\nTotal merges: {len(merges)}")
    print(f"First ten merges: {merges[:10]}")
except (NotImplementedError, RuntimeError) as err:
    print(f"Merge loop not runnable yet: {err}")


In [ ]:
"""§5 Subtask 4 (exercise): encode unseen text with the learned merges.

Walk each word, split it into characters + end-of-word marker, then replay every
merge rule in training order. Each rule contracts adjacent matching symbols.
"""


def encode(text: str, merges: list) -> list:
    # TODO: implement.
    raise NotImplementedError


probe = "the quick brown fox"
try:
    merges  # noqa: F821
except NameError:
    print("Run subtask 3 first to produce `merges`.")
else:
    if not merges:
        print("`merges` is empty; complete subtask 3 first.")
    else:
        try:
            print(f"Encoding of {probe!r}:")
            print(encode(probe, merges))
        except NotImplementedError:
            print("encode not implemented yet.")


In [ ]:
"""§5 Subtask 5 (exercise): compare to GPT-2 on the same sentence.

Once `encode` works, compare its output to GPT-2's tokenizer on the same
sentence. The comparison only runs when subtasks 3 and 4 are complete.
"""

sentence = "the quick brown fox jumps over the lazy dog"

try:
    merges  # noqa: F821
except NameError:
    print("Run subtask 3 first to produce `merges`.")
else:
    if not merges:
        print("`merges` is empty; complete subtask 3 first.")
    else:
        try:
            our_tokens = encode(sentence, merges)
        except NotImplementedError:
            our_tokens = None
            print("encode not implemented yet; complete subtask 4 first.")

        if our_tokens is not None:
            gpt2_tokens = gpt2_tok.tokenize(sentence)
            print(f"Sentence: {sentence!r}\n")
            print(f"Our BPE  ({len(merges)} merges trained on Shakespeare):")
            print(f"  {len(our_tokens):3d} tokens  {our_tokens}")
            print()
            print(f"GPT-2    (~50,000 merges trained on WebText):")
            print(f"  {len(gpt2_tokens):3d} tokens  {gpt2_tokens}")
            print()
            print("Reflect: why is GPT-2's count so much smaller? Two factors to weigh:")
            print("the merge count (~500x larger) and the byte-level pre-tokenisation.")


## §6 Recap and next step

You now have BPE in your hands: a corpus, an initial segmentation, a hundred merges, an encoder that replays those merges on unseen text, and a side-by-side comparison with a production-scale tokenizer. The vocabulary-versus-sequence-length tradeoff, the open-vocabulary problem, and the role of merge rules are no longer abstract.

From Session 1b onward the seminar calls `AutoTokenizer.from_pretrained(...)` and stops re-implementing this layer. The next notebook turns the integer IDs that come out of the tokenizer into vectors, which is what the rest of the transformer actually operates on.
